In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score
from sklearn.model_selection import StratifiedKFold
import lightgbm as lgb
import xgboost as xgb
import catboost as cb
from ipynb.fs.defs.preprocessing import get_data  # type: ignore

In [2]:
X_train, X_val, X_test, y_train, y_val, y_test = get_data(
    use_val=True, test_size=0.15, val_size=0.15
)
# X_train, X_test, y_train, y_test = get_data(use_val=False, test_size=0.3)

In [3]:
X_train

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount
124307,77251.0,-0.808407,0.482694,1.404276,0.199061,0.841043,0.134623,0.563333,0.283607,-0.349404,...,-0.150710,0.055123,0.152992,-0.168297,-0.335254,0.074801,-0.500801,0.113205,0.117894,2.901422
96760,65924.0,-0.387407,0.665193,1.788240,-0.069689,0.156406,-0.616384,0.855692,-0.228060,-0.347314,...,0.026085,0.032327,0.226153,-0.032060,0.414466,-0.486162,0.295972,-0.076663,-0.089357,3.000222
148174,89515.0,-0.773601,1.129779,-0.228021,0.798991,1.417614,-0.518593,1.152066,-0.186654,-0.700511,...,0.187950,0.119650,0.739064,-0.331333,-0.356161,0.113392,-0.380188,0.405320,0.374315,2.460443
68213,52883.0,-0.288436,1.019560,1.455446,0.040335,0.235227,-0.957739,0.940810,-0.318372,-0.320151,...,0.242815,-0.302088,-0.626548,-0.034506,0.346556,-0.166095,0.041582,0.081197,-0.122327,2.382320
141939,84531.0,1.245687,-0.498706,0.865425,-0.706149,-1.392459,-0.871495,-0.713169,0.084716,2.135298,...,-0.258264,-0.036487,0.043765,-0.011444,0.347270,0.430807,-0.679319,0.080104,0.030796,0.095310
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
60632,49418.0,-0.754280,0.228541,1.226787,-1.861144,0.038796,1.766069,-1.389738,-1.817258,-1.382534,...,-1.093396,1.922602,-1.395082,-0.161189,-1.026335,0.785869,1.055012,0.151478,0.124221,2.853016
178556,123681.0,1.953015,-0.356287,-1.878786,0.304870,0.277949,-0.816015,0.396964,-0.233897,0.339935,...,-0.162055,0.009867,0.001472,-0.035843,-0.462775,0.144918,0.568139,-0.121800,-0.083191,4.249780
229298,145870.0,-1.831394,1.828229,-1.826211,-1.737012,1.419505,-1.013023,1.637354,-0.426707,1.225536,...,0.560167,-0.130374,0.633117,-0.141900,-1.106015,0.141534,0.151035,0.932094,0.408407,2.054124
229666,146011.0,0.033308,0.802995,0.285098,-0.572603,0.358675,-1.158960,0.952132,-0.167765,0.059852,...,-0.080375,-0.266976,-0.626425,0.079982,0.020794,-0.513155,0.140053,0.242462,0.096303,0.867100


In [ ]:
N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
oof_preds = np.zeros(len(X_train))
best_iters_lgb, best_iters_xgb, best_iters_catm = [], [], []
for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    print(f"\n--- Fold {fold + 1}/{N_FOLDS} ---")
    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
    SCALE = np.bincount(y_tr)[0] / np.bincount(y_tr)[1]
    lgbm = lgb.LGBMClassifier(
        objective="binary",
        metric="average_precision",
        num_leaves=31,
        learning_rate=0.05,
        n_estimators=5000,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=SCALE,
        random_state=42 + fold,
        n_jobs=-1,
        verbose=-1,
    )
    lgbm.fit(
        X_tr,
        y_tr,
        # eval_set=[(X_val, y_val)],
        eval_X=X_val,
        eval_y=y_val,
        callbacks=[lgb.early_stopping(100), lgb.log_evaluation(1)],
    )
    xgbm = xgb.XGBClassifier(
        objective="binary:logistic",
        eval_metric="aucpr",
        max_depth=6,
        learning_rate=0.05,
        n_estimators=5000,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=SCALE,
        random_state=42 + fold,
        early_stopping_rounds=100,
        n_jobs=-1,
    )
    xgbm.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=True)
    catm = cb.CatBoostClassifier(
        iterations=5000,
        learning_rate=0.05,
        depth=6,
        eval_metric="PRAUC",
        auto_class_weights="Balanced",
        random_seed=42 + fold,
        early_stopping_rounds=100,
    )
    catm.fit(X_tr, y_tr, eval_set=(X_val, y_val), verbose=True)
    pred_val = (
        0.5 * lgbm.predict_proba(X_val)[:, 1]
        + 0.3 * xgbm.predict_proba(X_val)[:, 1]
        + 0.2 * catm.predict_proba(X_val)[:, 1]
    )
    oof_preds[val_idx] = pred_val
    best_iters_lgb.append(lgbm.best_iteration_)
    best_iters_xgb.append(xgbm.best_iteration)
    best_iters_catm.append(catm.best_iteration_)
    print(f"AUPRC fold {fold + 1}: {average_precision_score(y_val, pred_val):.6f}")
print(f"\n>>> OOF AUPRC 5-Fold: {average_precision_score(y_train, oof_preds):.6f}")



--- Fold 1/5 ---
[1]	valid_0's average_precision: 0.48828
Training until validation scores don't improve for 100 rounds
[2]	valid_0's average_precision: 0.0995843
[3]	valid_0's average_precision: 0.441617
[4]	valid_0's average_precision: 0.44251
[5]	valid_0's average_precision: 0.443593
[6]	valid_0's average_precision: 0.434112
[7]	valid_0's average_precision: 0.425735
[8]	valid_0's average_precision: 0.407352
[9]	valid_0's average_precision: 0.410648
[10]	valid_0's average_precision: 0.404134
[11]	valid_0's average_precision: 0.409251
[12]	valid_0's average_precision: 0.405049
[13]	valid_0's average_precision: 0.427941
[14]	valid_0's average_precision: 0.414648
[15]	valid_0's average_precision: 0.418449
[16]	valid_0's average_precision: 0.414121
[17]	valid_0's average_precision: 0.413472
[18]	valid_0's average_precision: 0.400534
[19]	valid_0's average_precision: 0.396884
[20]	valid_0's average_precision: 0.410937
[21]	valid_0's average_precision: 0.41207
[22]	valid_0's average_preci

In [ ]:
AVG_LGB = int(np.mean(best_iters_lgb))
AVG_XGB = int(np.mean(best_iters_xgb))
AVG_CATM = int(np.mean(best_iters_catm))
print(f"\nRetrain trên full data với n_estimators LGB={AVG_LGB}, XGB={AVG_XGB}")
neg, pos = np.bincount(y_train)
SCALE = neg / pos
final_lgbm = lgb.LGBMClassifier(
    objective="binary",
    metric="average_precision",
    num_leaves=31,
    learning_rate=0.03,
    n_estimators=2 * AVG_LGB,  # lr nhỏ hơn khi train full
    subsample=0.9,
    colsample_bytree=0.9,
    scale_pos_weight=SCALE,
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)
final_lgbm.fit(X_train, y_train)
final_xgbm = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric="aucpr",
    max_depth=6,
    learning_rate=0.03,
    n_estimators=2 * AVG_XGB,
    subsample=0.9,
    colsample_bytree=0.9,
    scale_pos_weight=SCALE,
    random_state=42,
    n_jobs=-1,
)
final_xgbm.fit(X_train, y_train, verbose=False)
final_catm = cb.CatBoostClassifier(
    learning_rate=0.03,
    depth=6,
    n_estimators=2 * AVG_CATM,
    eval_metric="PRAUC",
    auto_class_weights="Balanced",
    random_seed=42,
    verbose=False,
)
final_catm.fit(X_train, y_train, verbose=False)
# Test trên holdout X_test mà nãy giờ chưa đụng tới
final_pred_test = (
    0 * final_lgbm.predict_proba(X_test)[:, 1]
    + 0 * final_xgbm.predict_proba(X_test)[:, 1]
    + 1 * final_catm.predict_proba(X_test)[:, 1]
)
print(
    f"\n>>> AUPRC FINAL MODEL [train trên toàn bộ train] trên X_test: {average_precision_score(y_test, final_pred_test):.6f}"
)


Retrain trên full data với n_estimators LGB=6, XGB=377

>>> AUPRC FINAL MODEL [train trên toàn bộ train] trên X_test: 0.769429


In [ ]:
from util import summary
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    average_precision_score,
    roc_auc_score,
    precision_recall_curve,
)

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

# model = xgb.XGBClassifier(
#     n_estimators=3000,  # Số lượng cây (dùng early stopping để tránh overfit)
#     learning_rate=0.1,  # Tốc độ học nhỏ để học sâu hơn
#     max_depth=4,  # Độ sâu vừa phải tránh overfit với dữ liệu imbalanced
#     subsample=0.8,  # Lấy mẫu dữ liệu theo hàng
#     colsample_bytree=0.8,  # Lấy mẫu đặc trưng theo cột
#     scale_pos_weight=scale_pos_weight_value,  # Xử lý mất cân bằng
#     eval_metric="aucpr",  # Tối ưu diện tích dưới đường PR Curve (quan trọng nhất!)
#     early_stopping_rounds=100,  # Dừng nếu sau 50 vòng PR-AUC không cải thiện
#     random_state=42,
#     n_jobs=-1,
#     tree_method="hist"
# )
model = xgb.XGBClassifier(
    n_estimators=1000,
    max_depth=12,
    learning_rate=0.02,
    subsample=0.8,
    colsample_bytree=0.4,
    missing=-1,
    eval_metric="auc",
    # USE CPU
    nthread=4,
    tree_method="hist",
    scale_pos_weight=scale_pos_weight,
    # USE GPU
    # tree_method="gpu_hist",
)

model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=50)
# model.fit(X_train, y_train)


y_probs = model.predict_proba(X_test)[:, 1]
y_preds = model.predict(X_test)
print("\n" + "=" * 40)
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_probs):.4f}")
print(f"PR-AUC Score (AUPRC): {average_precision_score(y_test, y_probs):.4f}")
print("=" * 40)
summary(y_test, y_preds)


[0]	validation_0-auc:0.91842
[50]	validation_0-auc:0.96047
[100]	validation_0-auc:0.96739
[150]	validation_0-auc:0.97144
[200]	validation_0-auc:0.97067
[250]	validation_0-auc:0.97093
[300]	validation_0-auc:0.96876
[350]	validation_0-auc:0.97043
[400]	validation_0-auc:0.97021
[450]	validation_0-auc:0.97085
[500]	validation_0-auc:0.97013
[550]	validation_0-auc:0.97048
[600]	validation_0-auc:0.97049
[650]	validation_0-auc:0.97108
[700]	validation_0-auc:0.97084
[750]	validation_0-auc:0.97112
[800]	validation_0-auc:0.97077
[850]	validation_0-auc:0.97072
[900]	validation_0-auc:0.97129
[950]	validation_0-auc:0.97208
[999]	validation_0-auc:0.97241

ROC-AUC Score: 0.9958
PR-AUC Score (AUPRC): 0.8945
accuracy_score 0.9996020785543748
recall_score: 0.8153846153846154
precision_score: 0.9137931034482759
f1_score 0.8617886178861789
COnfusion matrix: [[42652     5]
 [   12    53]]
